# 循环神经网络


## 环境配置


In [ ]:
import os
import sys
sys.path.insert(0, "..")
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="Cannot create tensor with internal format")
import pypto
import torch
from torch import nn
import torch_npu
import logging
logging.getLogger('matplotlib').setLevel(logging.WARNING)

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()

## 练习8.4.1

**题目：** 如果我们使用循环神经网络来预测文本序列中的下一个字符，那么任意输出所需的维度是多少？

**解答：**

根据循环神经网络隐藏层与输出层的计算公式：

$$H_t = \phi(X_t W_{xh} + H_{t-1} W_{hh} + b_h)$$

$$O_t = H_t W_{hq} + b_q$$

其中 $X_t \in \mathbb{R}^{n \times d}$ 为第 $t$ 个时间步的输入（$n$ = 批量大小，$d$ = 输入维度），$H_t \in \mathbb{R}^{n \times h}$ 为第 $t$ 个时间步的隐状态（$h$ = 隐藏单元数量），$W_{xh} \in \mathbb{R}^{d \times h}$、$W_{hh} \in \mathbb{R}^{h \times h}$、$W_{hq} \in \mathbb{R}^{h \times q}$ 为权重矩阵，$\phi$ 为激活函数。

对于字符级语言模型（预测文本序列中的下一个字符）：每个时间步的输入为一个 one-hot 向量，其维度 = 词表大小 = $N$。隐状态 $H_t$ 维度为 (批量大小 $\times$ h)，输出 $O_t$ 维度为 (批量大小 $\times$ q)。由于输入和输出均来自同一词表，$q = N$，即任意输出所需的维度 = 词表大小。


以下使用 `torch` 编程进行验证：


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

batch_size, num_steps = 32, 35
_, vocab = d2l.load_data_time_machine(batch_size, num_steps)

X = torch.arange(10, device=device).reshape((2, 5))
print(f'X one_hot shape: {F.one_hot(X.T, 28).shape}')
# 输出: torch.Size([5, 2, 28]) —> (时间步, 批量大小, 词表大小)

def get_params(vocab_size, num_hiddens, device):
    num_inputs = num_outputs = vocab_size
    def normal(shape):
        return torch.randn(size=shape, device=device) * 0.01
    W_xh = normal((num_inputs, num_hiddens))
    W_hh = normal((num_hiddens, num_hiddens))
    b_h = torch.zeros(num_hiddens, device=device)
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)
    params = [W_xh, W_hh, b_h, W_hq, b_q]
    for param in params:
        param.requires_grad_(True)
    return params

def init_rnn_state(batch_size, num_hiddens, device):
    return (torch.zeros((batch_size, num_hiddens), device=device),)

def rnn(inputs, state, params):
    W_xh, W_hh, b_h, W_hq, b_q = params
    H, = state
    outputs = []
    for X in inputs:
        H = torch.tanh(torch.mm(X, W_xh) + torch.mm(H, W_hh) + b_h)
        Y = torch.mm(H, W_hq) + b_q
        outputs.append(Y)
    Y_stack = torch.stack(outputs)
    print(f'Y 堆叠后形状: {Y_stack.shape}')  # (时间步, 批量大小, 词表大小)
    return torch.cat(outputs, dim=0), (H,)

class RNNModelScratch:
    def __init__(self, vocab_size, num_hiddens, device,
                 get_params, init_state, forward_fn):
        self.vocab_size, self.num_hiddens = vocab_size, num_hiddens
        self.params = get_params(vocab_size, num_hiddens, device)
        self.init_state, self.forward_fn = init_state, forward_fn

    def __call__(self, X, state):
        X = F.one_hot(X.T, self.vocab_size).type(torch.float32)
        print(f'X 独热编码后形状: {X.shape}')
        return self.forward_fn(X, state, self.params)

    def begin_state(self, batch_size, device):
        return self.init_state(batch_size, self.num_hiddens, device)

num_hiddens = 256
net = RNNModelScratch(len(vocab), num_hiddens, d2l.try_gpu(),
                       get_params, init_rnn_state, rnn)
state = net.begin_state(X.shape[0], d2l.try_gpu())
Y, new_state = net(X.to(d2l.try_gpu()), state)
print(f'隐状态形状: {new_state[0].shape}')
print(f'输出形状: {Y.shape}')

使用 `PyPTO` 编程进行验证：


In [2]:
from src.utils import load_data_time_machine  # 本章节共享工具函数
from src.pypto_ops import PyPTOMatmul, PyPTOBiasAdd, PyPTOAdd, PyPTOTanh  # 本章节共享 PyPTO 算子

batch_size, num_steps = 32, 35
_, vocab = load_data_time_machine(batch_size, num_steps)
vocab_size = len(vocab)

X = torch.arange(10, device=device).reshape((2, 5))
X_oh = torch.nn.functional.one_hot(X.T, vocab_size).float()
print(f'X one_hot shape: {X_oh.shape}')
# 输出(预期): (时间步, 批量大小, 词表大小)

def init_rnn_state_pypto(batch_size, num_hiddens, device):
    return (torch.zeros((batch_size, num_hiddens), device=device),)

def rnn_forward_pypto(inputs, state, params):
    W_xh, W_hh, b_h, W_hq, b_q = params
    (H,) = state
    outputs = []
    for X_t in inputs:
        XW = PyPTOMatmul.apply(X_t, W_xh)
        HW = PyPTOMatmul.apply(H, W_hh)
        summed = PyPTOAdd.apply(XW, HW)
        biased = PyPTOBiasAdd.apply(summed, b_h)
        H = PyPTOTanh.apply(biased)
        Y = PyPTOBiasAdd.apply(PyPTOMatmul.apply(H, W_hq), b_q)
        outputs.append(Y)
    Y_stack = torch.stack(outputs)
    print(f'Y 堆叠后形状: {Y_stack.shape}')  # (时间步, 批量大小, 词表大小)
    return torch.cat(outputs, dim=0), (H,)

num_hiddens = 256
def normal(shape):
    return torch.randn(size=shape, device=device) * 0.01
W_xh = normal((vocab_size, num_hiddens))
W_hh = normal((num_hiddens, num_hiddens))
b_h = torch.zeros(num_hiddens, device=device)
W_hq = normal((num_hiddens, vocab_size))
b_q = torch.zeros(vocab_size, device=device)
params = [W_xh, W_hh, b_h, W_hq, b_q]
for param in params: param.requires_grad_(True)

state0 = init_rnn_state_pypto(X.shape[0], num_hiddens, device)
Y, new_state = rnn_forward_pypto(X_oh, state0, params)
print(f'隐状态形状: {new_state[0].shape}')  # (2, 256)
print(f'Y 形状: {Y.shape}')  # (时间步×批量, 词表大小)
print(f'可见输出维度 = 词表大小({vocab_size})，与输入维度一致')

X one_hot shape: torch.Size([5, 2, 28])


Y 堆叠后形状: torch.Size([5, 2, 28])
隐状态形状: torch.Size([2, 256])
Y 形状: torch.Size([10, 28])
可见输出维度 = 词表大小(28)，与输入维度一致


## 练习8.4.2

**题目：** 为什么循环神经网络可以基于文本序列中所有先前的词元，在某个时间步表示当前词元的条件概率？

**解答：**

语言模型的目标是根据过去和当前的词元来预测下一个词元。RNN 通过隐状态的递推关系 $H_t = f(H_{t-1}, X_t)$ 将历史信息编码进隐状态中，将其沿时间维度展开：

$$H_t = f(f(\dots f(H_0, X_1), X_2), \dots, X_t)$$

因此 $H_t$ 编码了从时间步 1 到 $t$ 的全部历史信息。输出 $O_t$ 由 $H_t$ 经线性变换加 softmax 得到：$O_t = \text{softmax}(H_t W_{hq} + b_q)$，因此 $O_t$ 实际上基于所有先前词元，能够建模条件概率 $P(x_t | x_1, \dots, x_{t-1})$。


## 练习8.4.3

**题目：** 如果基于一个长序列进行反向传播，梯度会发生什么状况？

**解答：**

BPTT 过程中梯度需沿时间步反向传播：$\partial L / \partial h_t = \partial L / \partial h_T \cdot \prod_{k=t}^{T-1} J_k$，其中 $J_k = \partial h_{k+1} / \partial h_k$。在平稳假设下 $J_k$ 近似为权重矩阵 $W$ 的某种变换，梯度传播相当于 $W^{T-t}$ 的连乘。

- 若梯度值普遍 < 1 → 连乘趋近 0 → **梯度消失**：前端时间步参数几乎得不到有效更新
- 若梯度值普遍 > 1 → 连乘发散 → **梯度爆炸**：参数更新过激，训练不稳定

这导致长序列训练中数值不稳定，模型难以捕获长距离依赖关系，这也是引入梯度裁剪和 LSTM/GRU 等门控机制的根本原因。


## 练习8.4.4

**题目：** 与本节中描述的语言模型相关的问题有哪些？

**解答：**

RNN 语言模型的应用场景：

1. **文本生成** — 根据前缀预测后续字符/词，通过对输出概率分布采样实现多样性生成（如 GPT 系列的自回归生成）。
2. **文本分类/情感分析** — 将序列最后一个时间步的隐状态输入全连接分类器进行二分类（正面/负面）或多分类。
3. **机器翻译 (seq2seq)** — 编码器 RNN 读入源语言序列生成上下文向量，解码器 RNN 基于上下文逐步生成目标语言序列。
4. **语音识别** — 将声学特征序列（如梅尔频谱）映射为文字序列，常配合 CTC 损失使用。
5. **命名实体识别 (NER)** — 对序列中每个词元进行标签预测（人名、地名、组织名等）。

PyPTO 的 RNN 实现同样可应用于上述各类语言模型任务，利用算子融合和自动 Tiling 进一步提升执行效率。


---
## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#/](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/)
